# 10 Report figures
By Tahreem Karim, Khalid Jackson, Chalse Okorom

Draws every new report figure from the CSVs written by notebooks 06-09, in Altair like
K-means infographic (Fig. B1). *Nothing here is for a model.*

| Figure | Shows | From |
|---|---|---|
| A1 | SHAP beeswarm & each feature group's share of attention | 06 |
| A2 | Hyperparameter plateau, & real policy vs state shuffled policy | 06 |
| A3 | Three failure cases, explained by fold models that scored them | 06 |
| B2 | K-means archetype profiles (one fitted model) | 08 |
| B3 | Archetypes on first two principal components | 08 |
| B4 | Gaussian mixture selection: BIC and silhouette by K | 08 |
| B5 | Archetype map, faded by membership uncertainty, w/ existing facilities | 08 |

#### Outputs `processed/figures/fig*.png` (2x scale).

In [ ]:
from types import SimpleNamespace

import altair as alt
import geopandas as gpd
import numpy as np
import pandas as pd
import vl_convert as vlc

from pipeline_config import FIGURES, GROUP, LABELS, ROOT, STAGE4, STAGE5, TARGET, load_model_table

alt.data_transformers.disable_max_rows()

# Same 'knobs' as infographic notebook's section 0.1 so new figures match Fig. B1
S = SimpleNamespace(
    font='Inter, system-ui, -apple-system, "Segoe UI", Helvetica, Arial, sans-serif',
    surface="#fcfcfb", ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9", axis="#c3c2b7",
    context="#c3c2b7", accent="#2a78d6", accent2="#eb6834", critical="#d03b3b",
    div_neg="#2a78d6", div_mid="#f0efec", div_pos="#e34948")

CLUSTER_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]
NAMES = ["Tax-light, pricey land", "Hot or dry, grid-remote", "Grid-close, cheap land"]


def ttl(text, subtitle=None):
    return alt.TitleParams(text=text, subtitle=subtitle or [], anchor="start")


def finish(chart):
    return (chart.configure(background=S.surface, font=S.font, padding=20)
            .configure_view(stroke=None)
            .configure_axis(labelColor=S.muted, titleColor=S.ink2, labelFontSize=11, titleFontSize=11.5,
                            titleFontWeight="normal", gridColor=S.grid, domainColor=S.axis, tickColor=S.axis)
            .configure_legend(labelColor=S.ink2, titleColor=S.ink2, labelFontSize=11, titleFontSize=11,
                              titleFontWeight="normal")
            .configure_title(fontSize=15, fontWeight=600, color=S.ink, subtitleColor=S.ink2, subtitleFontSize=12,
                             anchor="start", offset=10, subtitlePadding=4)
            .configure_header(labelColor=S.ink, labelFontSize=12.5, labelFontWeight=600, labelAnchor="start",
                              titleFontSize=0))


def save(chart, name):
    png = vlc.vegalite_to_png(finish(chart).to_json(), scale=2)
    (FIGURES / name).write_bytes(png)
    print("wrote", name)


d = load_model_table()

## Fig. A1 SHAP beeswarm & group shares
Draws SHAP beeswarm for top 12 features (each dot a county, colored by that county’s actual value) next to bar chart of each feature group’s total share of attention (visual version of importance tables from earlier).

In [ ]:
sh = pd.read_csv(STAGE4 / "shap_full.csv", dtype={"county_fips": str})
imp = pd.read_csv(STAGE4 / "shap_importance.csv", index_col=0)
share = pd.read_csv(STAGE4 / "shap_group_share.csv", index_col=0).share
top = imp.index[:12].tolist()
rows = []

for feat in top:
    s, v = sh[f"shap__{feat}"].values, sh[f"value__{feat}"].values
    pctl = pd.Series(v).rank(pct=True).values                       # color feature's value as percentile
    bins = np.digitize(s, np.histogram_bin_edges(s, 70))
    off = np.zeros(len(s))
    
    for b in np.unique(bins):                                       # spread points sharing x bin (a beeswarm)
        idx = np.where(bins == b)[0]
        idx = idx[np.argsort(np.random.default_rng(0).random(len(idx)))]
        off[idx] = (np.arange(len(idx)) - (len(idx) - 1) / 2)
    
    off = off / max(np.abs(off).max(), 1) * 0.42
    rows.append(pd.DataFrame({"feature": LABELS[feat], "shap": s, "pctl": pctl, "off": off}))

bee = pd.concat(rows)
order = [LABELS[f] for f in top]

swarm = alt.Chart(bee).mark_circle(size=9, opacity=0.75).encode(
    x=alt.X("shap:Q", title="SHAP value (pushes the log-odds of hosting a data center up or down)"),
    y=alt.Y("feature:N", sort=order, title=None, axis=alt.Axis(labelLimit=260, labelColor=S.ink2, labelFontSize=11.5)),
    yOffset=alt.YOffset("off:Q", scale=alt.Scale(domain=[-0.5, 0.5]), title=None),
    color=alt.Color("pctl:Q", scale=alt.Scale(domain=[0, 0.5, 1], range=[S.div_neg, "#d9d8d2", S.div_pos], interpolate="lab"),
                    legend=alt.Legend(title="Feature value (percentile)", orient="bottom", format="%", gradientLength=160)),
).properties(width=430, height=12 * 30)

zero = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(color=S.axis).encode(x="x:Q")
gs = share.rename("share").reset_index().rename(columns={"index": "group"})
gs["is_policy"] = gs.group.eq("AI policy")

bars = alt.Chart(gs).mark_bar(cornerRadiusEnd=4, height=16).encode(
    x=alt.X("share:Q", axis=None, scale=alt.Scale(domain=[0, gs.share.max() * 1.35])),
    y=alt.Y("group:N", sort="-x", title=None, axis=alt.Axis(labelColor=S.ink2, labelFontSize=11.5, domain=False, ticks=False)),
    color=alt.condition("datum.is_policy", alt.value(S.accent2), alt.value(S.context)))

labels = bars.mark_text(align="left", dx=5, fontSize=11.5).encode(text=alt.Text("share:Q", format=".0%"), color=alt.value(S.ink2))
right = (bars + labels).properties(width=170, height=6 * 34, title=ttl("Share of total |SHAP|", "by feature group"))
pol = share.get("AI policy", 0)

figA1 = alt.hconcat((zero + swarm).properties(title=ttl("Each dot is a county", "Top 12 features by mean |SHAP|; red = high value, blue = low")),
                    right, spacing=36).properties(
    title=ttl("Figure A1. Land value and grid access drive the model; AI policy gets " + f"{pol:.0%} of its attention, "
              "but removing it costs almost nothing out of sample",
              "XGBoost, all 29 features, fit on all 3,099 counties with the modal hyperparameters from nested CV"))

save(figA1, "figA1_shap.png") # Output

wrote figA1_shap.png


## Fig. A2 Hyperparameter plateau & permutation test
1. **Heatmap** of PR-AUC across tree depth × learning rate
2. **Histogram** of 100 "shuffled policy" permutation results w/ real result marked (to see how close to noise)

In [ ]:
hg = pd.read_csv(STAGE4 / "hyperparameter_grid.csv")
hg["label"] = hg["mean"].map(lambda v: f"{v:.3f}")
lo, hi = hg["mean"].min(), hg["mean"].max()

heat = alt.Chart(hg).mark_rect(stroke=S.surface, strokeWidth=2).encode(
    x=alt.X("learning_rate:O", title="Learning rate", axis=alt.Axis(labelAngle=0)), y=alt.Y("max_depth:O", title="Max tree depth"),
    color=alt.Color("mean:Q", scale=alt.Scale(range=["#eef3fb", "#0d366b"], domain=[lo - 0.01, hi]), legend=None))

heat_txt = alt.Chart(hg).mark_text(fontSize=11).encode(
    x="learning_rate:O", y="max_depth:O", text="label:N",
    color=alt.condition(f"datum.mean > {(lo + hi) / 2}", alt.value("white"), alt.value(S.ink)))

panel_a = (heat + heat_txt).properties(width=260, height=230, title=ttl("(a) PR-AUC by depth and learning rate",
                                                                         ["XGBoost, 500 trees, 15 state-grouped folds",
                                                                          "Depth barely matters; slower learning is better"]))
null = pd.read_csv(STAGE4 / "permutation_test.csv")
perm = pd.read_csv(STAGE4 / "permutation_summary.csv").set_index("family")
xg = null[null.family == "XGBoost"]
obs, pval = perm.loc["XGBoost", "observed_gain"], perm.loc["XGBoost", "p_value"]

hist = alt.Chart(xg).mark_bar(color=S.context, cornerRadiusEnd=2).encode(
    x=alt.X("gain:Q", bin=alt.Bin(maxbins=24), title="PR-AUC gain over + Geography"),
    y=alt.Y("count():Q", title="Shuffled policy profiles"))

vline = alt.Chart(pd.DataFrame({"x": [obs], "t": [f"real policy features {obs:+.3f} (p = {pval:.2f})"]}))
obs_rule = vline.mark_rule(color=S.accent2, strokeWidth=2.5).encode(x="x:Q")
obs_txt = vline.mark_text(align="left", dx=6, dy=-6, color=S.ink, fontSize=11.5, fontWeight=600).encode(
    x="x:Q", y=alt.value(12), text="t:N")

zero2 = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(color=S.muted, strokeDash=[3, 3]).encode(x="x:Q")
panel_b = (hist + zero2 + obs_rule + obs_txt).properties(width=300, height=230, title=ttl(
    "(b) Real policy vs 100 state-shuffled copies", ["Each copy gives every state another state's policy profile",
                                                     "XGBoost, same folds and hyperparameters"]))
figA2 = alt.hconcat(panel_a, panel_b, spacing=50).properties(title=ttl(
    "Figure A2. Slow learning rates form a plateau; real policy features are not reliably better than shuffled ones"))

save(figA2, "figA2_sensitivity.png"). # Output

wrote figA2_sensitivity.png


## Fig. A3 Three failure cases
Draws SHAP bar chart for each of three handpicked failure cases, showing top drivers behind that specific county’s prediction + caption w/ key numbers

In [ ]:
cases = pd.read_csv(STAGE4 / "failure_cases.csv", dtype={"county_fips": str})
fs = pd.read_csv(STAGE4 / "failure_shap.csv", dtype={"county_fips": str})
M = float(np.ceil(fs.shap.abs().max() * 2) / 2) + 0.25                 # one x scale for all three panels

NOTE = {
    "Confident false positive": lambda c: f"No atlas facility. Predicted {c.p:.0%} ({c.p_infra:.0%} from infrastructure alone)",
    "Missed campus": lambda c: f"{int(c.atlas_record_count)} atlas facilities ({c.operators} campuses). Predicted {c.p:.1%}",
    "Policy moved it wrong way": lambda c: f"{int(c.atlas_record_count)} atlas facilities. {c.p_geo:.0%} w/ region, "
                                               f"{c.p:.0%} once state AI policy added"}

panels = []
for _, c in cases.iterrows():
    g = fs[fs.county_fips == c.county_fips].assign(a=lambda t: t.shap.abs()).sort_values("a", ascending=False)
    head, rest = g.head(6), g.iloc[6:]
    rows = head[["feature", "shap"]].assign(feature=lambda t: t.feature.map(LABELS)).to_dict("records")
    rows.append({"feature": f"{len(rest)} other features", "shap": rest.shap.sum()})
    t = pd.DataFrame(rows)
    t["dir"] = np.where(t.shap > 0, "raises the odds", "lowers the odds")
    
    b = alt.Chart(t).mark_bar(cornerRadiusEnd=3, height=13).encode(
        x=alt.X("shap:Q", title="SHAP (log-odds, uncalibrated model)", scale=alt.Scale(domain=[-M, M]),
                axis=alt.Axis(values=list(range(-int(M), int(M) + 1)), format="d")),
        y=alt.Y("feature:N", sort=t.feature.tolist(), title=None, axis=alt.Axis(labelLimit=230, labelColor=S.ink2)),
        color=alt.Color("dir:N", scale=alt.Scale(domain=["raises the odds", "lowers the odds"], range=[S.div_pos, S.div_neg]),
                        legend=alt.Legend(title=None, orient="bottom")))
    
    z = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(color=S.axis).encode(x="x:Q")
    panels.append((z + b).properties(width=300, height=7 * 21, title=ttl(
        f"{c.case}: {c.county_name.title()} County, {c[GROUP]}", [NOTE[c.case](c)])))

figA3 = alt.vconcat(*panels, spacing=22).resolve_scale(color="shared").properties(title=ttl(
    "Figure A3. Three ways the model fails",
    "Out of fold SHAP vals avg'd over three fold models that held out each county's state"))

save(figA3, "figA3_failures.png") # Output

wrote figA3_failures.png


## Fig. B2 Archetype profiles (one fitted model)
Heatmap of each archetype’s 11 features, shown as SD from all county mean, so defining traits of cluster visible.

In [ ]:
prof = pd.read_csv(STAGE5 / "kmeans_profiles_sd.csv", index_col=0)
sizes = pd.read_csv(STAGE5 / "clusters.csv").kmeans.value_counts().sort_index()
prof.columns = [f"{k} · {NAMES[k]} ({sizes[k]:,})" for k in range(3)]
long = prof.rename(index=LABELS).reset_index(names="feature").melt("feature", var_name="cluster", value_name="sd")
order = prof.rename(index=LABELS).abs().max(axis=1).sort_values(ascending=False).index.tolist()
long["label"] = long.sd.map(lambda v: f"{v:+.1f}")

cells = alt.Chart(long).mark_rect(stroke=S.surface, strokeWidth=2).encode(
    x=alt.X("cluster:N", title=None, axis=alt.Axis(orient="top", labelAngle=0, labelLimit=220, labelColor=S.ink, labelFontSize=11.5)),
    y=alt.Y("feature:N", sort=order, title=None, axis=alt.Axis(labelColor=S.ink2, labelFontSize=11.5, labelLimit=240)),
    color=alt.Color("sd:Q", scale=alt.Scale(domain=[-2, 0, 2], range=[S.div_neg, S.div_mid, S.div_pos], clamp=True, interpolate="lab"),
                    legend=alt.Legend(title="SD from the all-county mean", orient="bottom", gradientLength=200)))

txt = alt.Chart(long).mark_text(fontSize=11).encode(
    x="cluster:N", y=alt.Y("feature:N", sort=order), text="label:N",
    color=alt.condition("abs(datum.sd) > 1.3", alt.value("white"), alt.value(S.ink)))

figB2 = (cells + txt).properties(width=3 * 190, height=11 * 30, title=ttl(
    "Figure B2. Each archetype is defined by one or two traits",
    "K-means, K = 3, 11 CERF features with electricity capped at the 99th percentile; cells clipped at ±2 SD for color"))

save(figB2, "figB2_profiles.png") # Output

wrote figB2_profiles.png


## Fig. Archetypes on two principal 
**Scatter** of every county on first two PCA components, colored by archetype, w/ a ring on counties that have an actual data center for a visual check of archetype separation.

In [ ]:
pc = pd.read_csv(STAGE5 / "pca_coords.csv")
ev = pd.read_csv(STAGE5 / "pca_variance.csv", index_col=0).iloc[:, 0]
pc["archetype"] = pc.kmeans.map(lambda k: f"{NAMES[k]} ({sizes[k]:,})")
dom = [f"{NAMES[k]} ({sizes[k]:,})" for k in range(3)]

pts = alt.Chart(pc).mark_circle(size=14, opacity=0.55).encode(
    x=alt.X("pc1:Q", title=f"PC1 ({ev['pc1']:.1%} of variance)"), y=alt.Y("pc2:Q", title=f"PC2 ({ev['pc2']:.1%} of variance)"),
    color=alt.Color("archetype:N", scale=alt.Scale(domain=dom, range=CLUSTER_COLORS), legend=alt.Legend(title=None, orient="top")))

ring = alt.Chart(pc[pc[TARGET] == 1].assign(k="Hosts a data center")).mark_point(size=26, strokeWidth=0.9, filled=False).encode(
    x="pc1:Q", y="pc2:Q", shape=alt.Shape("k:N", legend=alt.Legend(title=None, orient="top")), color=alt.value(S.ink))

figB3 = (pts + ring).properties(width=560, height=330, title=ttl(
    "Figure B3. Archetypes overlap at edges, data center counties sit in all three",
    f"First two principal components of same 11 standardized features ({ev.sum():.1%} of variance)"))

save(figB3, "figB3_pca.png"). #Output

wrote figB3_pca.png


## Fig. B4 Gaussian mixture selection
Two panels showing why K=3 kept over BIC optimal K. 

In [ ]:
sw = pd.read_csv(STAGE5 / "gmm_sweep.csv")
b1 = pd.read_csv(STAGE5 / "table_b1.csv")

km_sil = b1.loc[b1.method.str.startswith("K-means"), "silhouette"].iloc[0]
sw["cov_label"] = sw.covariance.map({"full": "full covariance", "diag": "diagonal"})
covc = alt.Color("cov_label:N", scale=alt.Scale(domain=["full covariance", "diagonal"], range=[S.accent, S.accent2]), legend=None)
ends = sw[sw.K == 10]

bic = alt.Chart(sw).mark_line(point=alt.OverlayMarkDef(size=40), strokeWidth=2).encode(
    x=alt.X("K:O", title="Components (K)", axis=alt.Axis(labelAngle=0)), y=alt.Y("bic:Q", title="BIC (lower is better)", scale=alt.Scale(zero=False)),
    color=covc)

bic_lab = alt.Chart(ends).mark_text(align="left", dx=8, fontSize=11).encode(x="K:O", y="bic:Q", text="cov_label:N", color=alt.value(S.ink2))
bic_panel = (bic + bic_lab).properties(width=270, height=220, title=ttl("(a) BIC keeps falling", "so its minimum is not a usable choice"))
sil = alt.Chart(sw).mark_line(point=alt.OverlayMarkDef(size=40), strokeWidth=2).encode(
    x=alt.X("K:O", title="Components (K)", axis=alt.Axis(labelAngle=0)), y=alt.Y("silhouette:Q", title="Silhouette of hard labels"),
    color=covc)

sil_lab = alt.Chart(ends).mark_text(align="left", dx=8, fontSize=11).encode(x="K:O", y="silhouette:Q", text="cov_label:N", color=alt.value(S.ink2))
ref = alt.Chart(pd.DataFrame({"y": [km_sil], "t": [f"K-means, K = 3 ({km_sil:.3f})"]}))
sil_panel = (sil + sil_lab + ref.mark_rule(color=S.muted, strokeDash=[4, 3]).encode(y="y:Q")
             + ref.mark_text(align="right", dy=-6, color=S.ink2, fontSize=11).encode(y="y:Q", x=alt.value(270), text="t:N")
             ).properties(width=270, height=220, title=ttl("(b) Separation stays below K-means", "silhouette on a 2,000-county sample"))

figB4 = alt.hconcat(bic_panel, sil_panel, spacing=90).properties(title=ttl(
    "Figure B4. Gaussian mixture selection, K = 2 to 10", "11 standardized CERF features, reg_covar = 1e-3, 5 initializations"))

save(figB4, "figB4_gmm.png") #Output

wrote figB4_gmm.png


## Fig. B5 Archetype map w/ membership uncertainty & existing facilities
US county map colored by archetype (paler where a county sits ambiguously between two), w/ dots marking counties that have a data center, sized by facility count.

In [ ]:
cl = pd.read_csv(STAGE5 / "clusters.csv", dtype={"county_fips": str})

shapes = gpd.read_file(ROOT / "raw data" / "census" / "cb_2024_us_county_500k")[["GEOID", "STUSPS", "geometry"]]
shapes = shapes[~shapes.STUSPS.isin(["AK", "HI", "PR", "VI", "GU", "MP", "AS"])].to_crs(5070)
shapes["geometry"] = shapes.geometry.simplify(2500)
states = shapes.dissolve("STUSPS").reset_index()[["STUSPS", "geometry"]]

m = shapes.merge(cl, left_on="GEOID", right_on="county_fips", how="left")
m["archetype"] = m.kmeans.map(lambda k: NAMES[int(k)] if pd.notna(k) else "Not modeled")
m["opacity"] = np.where(m.kmeans.notna(), 1 - 0.75 * m.kmeans_fit_ratio.fillna(0) ** 2, 1)   # paler = nearly as close to a second center
m = m.to_crs(4326)
m["geometry"] = m.geometry.orient_polygons(exterior_cw=True)

cent = shapes.to_crs(4326).set_index("GEOID").geometry.representative_point()
dots = d[d[TARGET] == 1][["county_fips", "atlas_record_count"]].assign(
    lon=lambda t: t.county_fips.map(cent.x), lat=lambda t: t.county_fips.map(cent.y))

base_map = alt.Chart(m).mark_geoshape(stroke=S.surface, strokeWidth=0.25).encode(
    color=alt.Color("archetype:N", scale=alt.Scale(domain=NAMES + ["Not modeled"], range=CLUSTER_COLORS + ["#e9e8e3"]),
                    legend=alt.Legend(title=None, orient="top", columns=4)),
    opacity=alt.Opacity("opacity:Q", legend=None, scale=None)).project(type="albersUsa")

states = states.to_crs(4326)
states["geometry"] = states.geometry.orient_polygons(exterior_cw=True)
outline = alt.Chart(states).mark_geoshape(fill=None, stroke="#ffffff", strokeWidth=0.9).project(type="albersUsa")

dc = alt.Chart(dots.assign(k="County with an atlas facility (size = facilities)")).mark_circle(
    color=S.ink, opacity=0.8, stroke=S.surface, strokeWidth=0.4).encode(
    longitude="lon:Q", latitude="lat:Q",
    size=alt.Size("atlas_record_count:Q", scale=alt.Scale(range=[6, 160], type="sqrt"),
                  legend=alt.Legend(title="Atlas facilities in the county", orient="top", values=[1, 10, 50, 200])),
).project(type="albersUsa")

figB5 = (base_map + outline + dc).properties(width=760, height=460, title=ttl(
    "Figure B5. Archetypes follow state tax lines & climate. Paler counties sit between two archetypes",
    ["K-means archetypes: paler = closer to a second center (distance ratio). Dots: counties with atlas facilities, sized by count.",
     "Gaussian mixture entropy not used for shading, mixture is near certain (entropy < 0.1) for 99% of counties."]))

save(figB5, "figB5_map.png") #Output

wrote figB5_map.png
